In [43]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE = "https://premiaruneta.ru"
START = BASE + "/news"                      # стартовая страница со списком
HEADERS = {"User-Agent": "Mozilla/5.0"}

def get(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.encoding = r.apparent_encoding
    return BeautifulSoup(r.text, "lxml")

def parse_article(url):
    s = get(url)
    h1 = s.find("h1")
    body = s.find("article") or s.find("main")
    text = "\n".join(p.get_text(strip=True) for p in body.find_all("p")) if body else ""
    date = s.find("time")
    return {
        "company_name":     h1.get_text(strip=True) if h1 else None,
        "description":      text[:300],
        "article_text":     text,
        "publication_date": (date.get("datetime") or date.text) if date else None,
        "industry":         None,           # заполняем ниже, если найдём номинацию
        "rating":           None,
        "source_url":       url,
        "scraped_at":       pd.Timestamp.utcnow(),
    }

def main():
    soup = get(START)

    # 1) ссылки на статьи со стартовой страницы
    links = [BASE + a["href"] if a["href"].startswith("/") else a["href"]
             for a in soup.select("a[href]") if "/news/" in a["href"]]
    links = list(dict.fromkeys(links))       # убираем дубли

    # 2) парсим каждую
    rows = []
    for i, url in enumerate(links, 1):
        print(f"[{i}/{len(links)}] {url}")
        try:
            rows.append(parse_article(url))
        except Exception as e:
            print("skip:", e)
        time.sleep(1)                        # вежливая пауза

    # 3) сохраняем
    df = pd.DataFrame(rows)
    df.to_csv("dataset.csv", index=False, encoding="utf-8-sig")
    df.to_json("dataset.json", orient="records", force_ascii=False, indent=2)
    print(f"Готово: {len(df)} строк")

if __name__ == "__main__":
    main()

[1/53] https://premiaruneta.ru/news/itogi-premii-runeta-pobeditel-v-nominatsii-tsifrovoe-nasledie-internet-entsiklopediya-ruviki
[2/53] https://premiaruneta.ru/news/proekt-akademiya-blogerov-dlya-avtorov-videohostinga-rutube-stal-pobeditelem-premii-runeta-v-nominatsii-blogerskiy-proekt-v-it
[3/53] https://premiaruneta.ru/news/itogi-premii-runeta-obyavleny-pobediteli-spetsialnoy-nominatsii-internet-na-rodnom-yazyke
[4/53] https://premiaruneta.ru/news/rostelekom-na-vysote-kompaniya-poluchila-dve-nagrady-premii-runeta2025
[5/53] https://premiaruneta.ru/news/premiya-runeta-2025-luchshie-it-proekty-rossii-nazvany-na-torzhestvennoy-tseremonii-v-dome-pashkova
[6/53] https://premiaruneta.ru/news/tenchat-informatsionnyy-partnyor-premii-runeta2025
[7/53] https://premiaruneta.ru/news/sostavru-partnyor-premii-runeta2025-kod-mechty
[8/53] https://premiaruneta.ru/news/gruppa-kompaniy-solar-partnyor-premii-runeta2025-kod-mechty
[9/53] https://premiaruneta.ru/news/mediagruppa-komsomolskaya-pravda-info

In [1]:
import pandas as pd

df = pd.read_csv("runet_nominees.csv", delimiter=";")
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 127 entries, 0 to 126
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                127 non-null    int64  
 1   company           127 non-null    object 
 2   description       127 non-null    object 
 3   sphere            127 non-null    object 
 4   rating            127 non-null    float64
 5   nomination        127 non-null    object 
 6   publication_date  127 non-null    object 
 7   article_title     127 non-null    object 
 8   author            127 non-null    object 
 9   hub               127 non-null    object 
 10  article_url       127 non-null    object 
 11  article_text      127 non-null    object 
 12  word_count        127 non-null    int64  
dtypes: float64(1), int64(2), object(10)
memory usage: 13.0+ KB


In [2]:
df.drop_duplicates()

,id,company,description,sphere,rating,nomination,publication_date,article_title,author,hub,article_url,article_text,word_count
0,1,Kaspersky,"Международная компания, разработчик систем защ...",Кибербезопасность,9.6,Технологии и инновации,2023-01-28,Почему Kaspersky претендует на звание лучшего ...,ml_engineer_k,Карьера в IT-индустрии,https://habr.com/ru/articles/858744/,Kaspersky — один из флагманов российских техно...,159
1,2,Ростелеком,"Крупнейший в России оператор связи, провайдер ...",Телекоммуникации и цифровые сервисы,9.3,Технологии и инновации,2023-01-28,"Разбираем кейс Ростелеком: технологии, команда...",habr_analytics,Анализ и проектирование систем,https://habr.com/ru/articles/720031/,Ростелеком&nbsp;— один из флагманов российских...,155
2,3,ФНС России,"Федеральная налоговая служба, внедрившая онлай...",Государственные информационные системы,9.3,Государство и общество,2023-02-02,"Разбираем кейс ФНС России: технологии, команда...",runet_observer,Машинное обучение,https://habr.com/ru/articles/739267/,Команда ФНС России внедряет принципы сервисног...,140
3,4,Фоксфорд,"Онлайн-школа с курсами, репетиторами и внеуроч...",Дополнительное образование,8.6,Образование и карьера,2023-02-05,Как Фоксфорд меняет рынок: анализ продуктов и ...,it_reviewer_24,Рост и развитие стартапов,https://habr.com/ru/articles/677432/,Фоксфорд — один из лидеров российского EdTech....,76
4,5,Яндекс,Крупнейшая российская технологическая компания...,"Поиск, ИИ и облачные технологии",9.8,Технологии и инновации,2023-02-13,Номинант Премии Рунета — Яндекс: что нужно зна...,ml_engineer_k,Интернет-маркетинг,https://habr.com/ru/articles/776472/,За последний год Яндекс представила несколько ...,162
...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,123,МЕДСИ,"Сеть частных клиник, активно внедряющая цифров...",Частная медицина и цифровизация,8.8,Здоровье и медицина,2025-10-17,Номинант Премии Рунета — МЕДСИ: что нужно знат...,digital_expert,Анализ и проектирование систем,https://habr.com/ru/articles/887731/,"Эксперты здравоохранения отмечают, что цифровы...",78
123,124,amoCRM,Простая облачная CRM-система для управления пр...,CRM и продажи,8.6,Цифровая экономика и бизнес,2025-10-22,Почему amoCRM претендует на звание лучшего в Р...,data_analyst_ru,Анализ и проектирование систем,https://habr.com/ru/articles/806580/,В этом году amoCRM сфокусировалась на развитии...,148
124,125,МойСклад,Облачный сервис управления торговлей: складско...,Автоматизация торговли,8.5,Цифровая экономика и бизнес,2025-10-24,Номинант Премии Рунета — МойСклад: что нужно з...,ml_engineer_k,Карьера в IT-индустрии,https://habr.com/ru/articles/687969/,"По оценкам аналитиков, рынок корпоративного ПО...",106
125,126,СберЗдоровье,Телемедицинский сервис: онлайн-консультации вр...,Телемедицина,9.2,Здоровье и медицина,2025-11-06,Почему СберЗдоровье претендует на звание лучше...,data_analyst_ru,Рост и развитие стартапов,https://habr.com/ru/articles/644212/,"Эксперты здравоохранения отмечают, что цифровы...",75


In [3]:
columns_to_remove = ['id', 'company', 'description', 'author', 'article_url']
df_clean = df.drop(columns=columns_to_remove)

df_clean.to_csv('structured_dataset.csv', index=False, sep=';', encoding='utf-8-sig')

In [4]:
df_clean

,sphere,rating,nomination,publication_date,article_title,hub,article_text,word_count
0,Кибербезопасность,9.6,Технологии и инновации,2023-01-28,Почему Kaspersky претендует на звание лучшего ...,Карьера в IT-индустрии,Kaspersky — один из флагманов российских техно...,159
1,Телекоммуникации и цифровые сервисы,9.3,Технологии и инновации,2023-01-28,"Разбираем кейс Ростелеком: технологии, команда...",Анализ и проектирование систем,Ростелеком&nbsp;— один из флагманов российских...,155
2,Государственные информационные системы,9.3,Государство и общество,2023-02-02,"Разбираем кейс ФНС России: технологии, команда...",Машинное обучение,Команда ФНС России внедряет принципы сервисног...,140
3,Дополнительное образование,8.6,Образование и карьера,2023-02-05,Как Фоксфорд меняет рынок: анализ продуктов и ...,Рост и развитие стартапов,Фоксфорд — один из лидеров российского EdTech....,76
4,"Поиск, ИИ и облачные технологии",9.8,Технологии и инновации,2023-02-13,Номинант Премии Рунета — Яндекс: что нужно зна...,Интернет-маркетинг,За последний год Яндекс представила несколько ...,162
...,...,...,...,...,...,...,...,...
122,Частная медицина и цифровизация,8.8,Здоровье и медицина,2025-10-17,Номинант Премии Рунета — МЕДСИ: что нужно знат...,Анализ и проектирование систем,"Эксперты здравоохранения отмечают, что цифровы...",78
123,CRM и продажи,8.6,Цифровая экономика и бизнес,2025-10-22,Почему amoCRM претендует на звание лучшего в Р...,Анализ и проектирование систем,В этом году amoCRM сфокусировалась на развитии...,148
124,Автоматизация торговли,8.5,Цифровая экономика и бизнес,2025-10-24,Номинант Премии Рунета — МойСклад: что нужно з...,Карьера в IT-индустрии,"По оценкам аналитиков, рынок корпоративного ПО...",106
125,Телемедицина,9.2,Здоровье и медицина,2025-11-06,Почему СберЗдоровье претендует на звание лучше...,Рост и развитие стартапов,"Эксперты здравоохранения отмечают, что цифровы...",75


In [5]:
df.isna().sum()

id                  0
company             0
description         0
sphere              0
rating              0
nomination          0
publication_date    0
article_title       0
author              0
hub                 0
article_url         0
article_text        0
word_count          0
dtype: int64

In [6]:
df.describe()

,id,rating,word_count
count,127.000000,127.000000,127.000000
mean,64.000000,9.092126,111.236220
std,36.805797,0.396334,26.242295
min,1.000000,8.200000,66.000000
25%,32.500000,8.800000,91.500000
50%,64.000000,9.100000,112.000000
75%,95.500000,9.400000,132.500000
max,127.000000,9.800000,162.000000


In [7]:
import pandas as pd
import re
import pymorphy3

data = df_clean
morph = pymorphy3.MorphAnalyzer()

def clean_text(text):
    if not isinstance(text, str): return ""
    words = re.findall(r'[а-яёa-z]+', text.lower())
    return " ".join([morph.parse(w)[0].normal_form for w in words])

data['cleaned_text'] = data['article_text'].apply(clean_text)



In [8]:
import pandas as pd
import re

def tokenize(text):
    if not isinstance(text, str): return []
    return re.findall(r'[а-яёa-z]+', text.lower())
data['tokens'] = data['article_text'].apply(tokenize)


In [9]:
import pandas as pd
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))

def remove_stopwords(text):
    if not isinstance(text, str): return ""
    return " ".join([w for w in text.split() if w not in russian_stopwords])

data['cleaned_text'] = data['cleaned_text'].apply(remove_stopwords)
data.to_csv('final_dataset.csv', index=False, sep=';', encoding='utf-8-sig')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
data

,sphere,rating,nomination,publication_date,article_title,hub,article_text,word_count,cleaned_text,tokens
0,Кибербезопасность,9.6,Технологии и инновации,2023-01-28,Почему Kaspersky претендует на звание лучшего ...,Карьера в IT-индустрии,Kaspersky — один из флагманов российских техно...,159,kaspersky флагман российский технология данные...,"[kaspersky, один, из, флагманов, российских, т..."
1,Телекоммуникации и цифровые сервисы,9.3,Технологии и инновации,2023-01-28,"Разбираем кейс Ростелеком: технологии, команда...",Анализ и проектирование систем,Ростелеком&nbsp;— один из флагманов российских...,155,ростелеком nbsp флагман российский технология ...,"[ростелеком, nbsp, один, из, флагманов, россий..."
2,Государственные информационные системы,9.3,Государство и общество,2023-02-02,"Разбираем кейс ФНС России: технологии, команда...",Машинное обучение,Команда ФНС России внедряет принципы сервисног...,140,команда фнс россия внедрять принцип сервисный ...,"[команда, фнс, россии, внедряет, принципы, сер..."
3,Дополнительное образование,8.6,Образование и карьера,2023-02-05,Как Фоксфорд меняет рынок: анализ продуктов и ...,Рост и развитие стартапов,Фоксфорд — один из лидеров российского EdTech....,76,фоксфорд лидер российский edtech рейтинг публи...,"[фоксфорд, один, из, лидеров, российского, edt..."
4,"Поиск, ИИ и облачные технологии",9.8,Технологии и инновации,2023-02-13,Номинант Премии Рунета — Яндекс: что нужно зна...,Интернет-маркетинг,За последний год Яндекс представила несколько ...,162,последний год яндекс представить несколько зам...,"[за, последний, год, яндекс, представила, неск..."
...,...,...,...,...,...,...,...,...,...,...
122,Частная медицина и цифровизация,8.8,Здоровье и медицина,2025-10-17,Номинант Премии Рунета — МЕДСИ: что нужно знат...,Анализ и проектирование систем,"Эксперты здравоохранения отмечают, что цифровы...",78,эксперт здравоохранение отмечать цифровой серв...,"[эксперты, здравоохранения, отмечают, что, циф..."
123,CRM и продажи,8.6,Цифровая экономика и бизнес,2025-10-22,Почему amoCRM претендует на звание лучшего в Р...,Анализ и проектирование систем,В этом году amoCRM сфокусировалась на развитии...,148,это год amocrm сфокусироваться развитие экосис...,"[в, этом, году, amocrm, сфокусировалась, на, р..."
124,Автоматизация торговли,8.5,Цифровая экономика и бизнес,2025-10-24,Номинант Премии Рунета — МойСклад: что нужно з...,Карьера в IT-индустрии,"По оценкам аналитиков, рынок корпоративного ПО...",106,оценка аналитик рынок корпоративный россия рас...,"[по, оценкам, аналитиков, рынок, корпоративног..."
125,Телемедицина,9.2,Здоровье и медицина,2025-11-06,Почему СберЗдоровье претендует на звание лучше...,Рост и развитие стартапов,"Эксперты здравоохранения отмечают, что цифровы...",75,эксперт здравоохранение отмечать цифровой серв...,"[эксперты, здравоохранения, отмечают, что, циф..."


In [11]:
import pandas as pd

# Создаем целевую переменную target и удаляем старую колонку nomination
data['target'] = data['nomination']
data = data.drop(columns=['nomination'])



In [12]:
data

,sphere,rating,publication_date,article_title,hub,article_text,word_count,cleaned_text,tokens,target
0,Кибербезопасность,9.6,2023-01-28,Почему Kaspersky претендует на звание лучшего ...,Карьера в IT-индустрии,Kaspersky — один из флагманов российских техно...,159,kaspersky флагман российский технология данные...,"[kaspersky, один, из, флагманов, российских, т...",Технологии и инновации
1,Телекоммуникации и цифровые сервисы,9.3,2023-01-28,"Разбираем кейс Ростелеком: технологии, команда...",Анализ и проектирование систем,Ростелеком&nbsp;— один из флагманов российских...,155,ростелеком nbsp флагман российский технология ...,"[ростелеком, nbsp, один, из, флагманов, россий...",Технологии и инновации
2,Государственные информационные системы,9.3,2023-02-02,"Разбираем кейс ФНС России: технологии, команда...",Машинное обучение,Команда ФНС России внедряет принципы сервисног...,140,команда фнс россия внедрять принцип сервисный ...,"[команда, фнс, россии, внедряет, принципы, сер...",Государство и общество
3,Дополнительное образование,8.6,2023-02-05,Как Фоксфорд меняет рынок: анализ продуктов и ...,Рост и развитие стартапов,Фоксфорд — один из лидеров российского EdTech....,76,фоксфорд лидер российский edtech рейтинг публи...,"[фоксфорд, один, из, лидеров, российского, edt...",Образование и карьера
4,"Поиск, ИИ и облачные технологии",9.8,2023-02-13,Номинант Премии Рунета — Яндекс: что нужно зна...,Интернет-маркетинг,За последний год Яндекс представила несколько ...,162,последний год яндекс представить несколько зам...,"[за, последний, год, яндекс, представила, неск...",Технологии и инновации
...,...,...,...,...,...,...,...,...,...,...
122,Частная медицина и цифровизация,8.8,2025-10-17,Номинант Премии Рунета — МЕДСИ: что нужно знат...,Анализ и проектирование систем,"Эксперты здравоохранения отмечают, что цифровы...",78,эксперт здравоохранение отмечать цифровой серв...,"[эксперты, здравоохранения, отмечают, что, циф...",Здоровье и медицина
123,CRM и продажи,8.6,2025-10-22,Почему amoCRM претендует на звание лучшего в Р...,Анализ и проектирование систем,В этом году amoCRM сфокусировалась на развитии...,148,это год amocrm сфокусироваться развитие экосис...,"[в, этом, году, amocrm, сфокусировалась, на, р...",Цифровая экономика и бизнес
124,Автоматизация торговли,8.5,2025-10-24,Номинант Премии Рунета — МойСклад: что нужно з...,Карьера в IT-индустрии,"По оценкам аналитиков, рынок корпоративного ПО...",106,оценка аналитик рынок корпоративный россия рас...,"[по, оценкам, аналитиков, рынок, корпоративног...",Цифровая экономика и бизнес
125,Телемедицина,9.2,2025-11-06,Почему СберЗдоровье претендует на звание лучше...,Рост и развитие стартапов,"Эксперты здравоохранения отмечают, что цифровы...",75,эксперт здравоохранение отмечать цифровой серв...,"[эксперты, здравоохранения, отмечают, что, циф...",Здоровье и медицина


SyntaxError: invalid syntax (3307521294.py, line 1)